In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

In [5]:
import json
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel
from evaluation_utils import llm_structured, map_progress

In [6]:
class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain The answer to each question.
- Make The questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- ask about The content of The lesson, not about its formatting or filename.
""".strip()

In [9]:
import inspect
from evaluation_utils import llm_structured

print(inspect.signature(llm_structured))

(client, instructions, user_prompt, output_type, model='gpt-5.4-mini')


In [10]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

In [11]:
page = documents[0]

user_prompt = json.dumps({
    "filename": page["filename"],
    "content": page["content"]
})

questions, tokens = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Questions
)

questions

Questions(questions=['What is the main idea behind retrieval-augmented generation, and why does it help with answers?', 'Why does this course use the LLM as a black box instead of explaining how it works inside?', 'What problems do large language models have that RAG is meant to fix?', 'What are the main things that Part 1 of this module will cover before the agentic version?', 'How will the final FAQ assistant in this module work, and what kind of questions is it meant to answer?'])

In [12]:
def generate_questions(page):
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"]
    })

    questions, tokens = llm_structured(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    return questions, tokens

In [14]:
token_counts = []

for page in documents[:3]:
    questions, tokens = generate_questions(page)

    token_counts.append(tokens.input_tokens)

    print(page["filename"], tokens.input_tokens)

print("Average:", sum(token_counts) / len(token_counts))

01-agentic-rag/lessons/01-intro.md 1021
01-agentic-rag/lessons/02-environment.md 1287
01-agentic-rag/lessons/03-rag.md 1754
Average: 1354.0


In [17]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth), ground_truth[0]

(360,
 {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'})

In [22]:
from minsearch import Index, VectorSearch

In [23]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [24]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [25]:
q = ground_truth[0]["question"]

text_results = text_search(q)

q, text_results[0]["filename"]

("What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 '01-agentic-rag/lessons/03-rag.md')

In [30]:
from embedder import Embedder
import numpy as np

embedder = Embedder()

2026-06-29 21:01:46.967367787 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [31]:
texts = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(texts)
X = np.array(X)

X.shape

(295, 384)

In [33]:
from minsearch import VectorSearch

vector_index = VectorSearch(keyword_fields=["filename"])

vector_index.fit(X, chunks)

In [34]:
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

In [35]:
q = ground_truth[0]["question"]

vector_results = vector_search(q)

vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [36]:
def compute_relevance(record, search_function):
    question = record["question"]
    target_filename = record["filename"]

    results = search_function(question)

    relevance = [
        int(result["filename"] == target_filename)
        for result in results
    ]

    return relevance


def hit_rate(relevance_total):
    cnt = 0

    for relevance in relevance_total:
        if 1 in relevance:
            cnt = cnt + 1

    return cnt / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0

    for relevance in relevance_total:
        for rank, rel in enumerate(relevance):
            if rel == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance_total)


def evaluate(ground_truth, search_function):
    relevance_total = []

    for record in ground_truth:
        relevance = compute_relevance(record, search_function)
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [37]:
evaluate(ground_truth, text_search)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [38]:
evaluate(ground_truth, vector_search)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [39]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [40]:
for k in [1, 50, 100, 200]:
    metrics = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )
    print(f"k={k}: {metrics}")

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}
k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
